# Overall Survival Prediction -- Acute Myeloid Leukemia
**QRT Data Challenge 2025**

---

## Context

**Acute Myeloid Leukemia (AML)** is a hematological cancer characterized by the rapid proliferation of immature blast cells in the bone marrow.

**Objective:** Estimate a risk score for overall survival (OS) using clinical and molecular data.

- `OS_YEARS`: follow-up time | `OS_STATUS`: 1=death, 0=censored

## Results summary

| Model | CV C-index |
|---|---|
| RSF -- Clinical only | 0.7138 |
| **RSF -- Clinical + Molecular** | **0.7419** |
| Final test platform | **0.7524** |

## Table of contents
1. [Setup & data loading](#1-setup)
2. [EDA -- Clinical](#2-eda-clinical)
3. [EDA -- Molecular](#3-eda-molecular)
4. [Preprocessing](#4-preprocessing)
5. [Modelling & evaluation](#5-modelling)
6. [Final fitting & submission](#6-final)


## 1. Setup & data loading <a id='1-setup'></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
import joblib

from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import IsolationForest

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test

from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

# Local utilities
from src.config import (
    CLINICAL_TRAIN, MOLECULAR_TRAIN, TARGET_TRAIN,
    CLINICAL_TEST, MOLECULAR_TEST,
    CLINICAL_RSF, CLINICAL_PIPELINE,
    FULL_RSF, FULL_PIPELINE, FULL_SELECTED_FEATURES, FULL_MOLECULAR_FEATURES,
    FIGURES_DIR, check_data_files, check_models
)
from src.features import cytogenetic_group, CONTINUOUS_VARS
from src.preprocessing import summarize_outliers

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

print('Setup complete.')

In [ ]:
check_data_files()

df_clinical  = pd.read_csv(CLINICAL_TRAIN)
df_molecular = pd.read_csv(MOLECULAR_TRAIN)
df_y         = pd.read_csv(TARGET_TRAIN)

print(f'Shapes -- clinical: {df_clinical.shape} | molecular: {df_molecular.shape} | target: {df_y.shape}')
print(f'Unique IDs -- clinical: {df_clinical["ID"].nunique()} | molecular: {df_molecular["ID"].nunique()}')

## 2. EDA -- Clinical data <a id='2-eda-clinical'></a>

### 2.1 Dataset overview

In [ ]:
df = df_clinical.merge(df_y, on='ID')
df_eda = df.copy()
df_eda['CYTO_CLASS'] = df_eda['CYTOGENETICS'].apply(cytogenetic_group)

CATEGORICAL_VARS = ['CENTER', 'CYTOGENETICS', 'OS_STATUS']
TARGET_TIME      = 'OS_YEARS'
TARGET_EVENT     = 'OS_STATUS'

print('Dataset shape:', df.shape)
display(df_clinical.info())
display(df.isna().sum().rename('missing'))

### 2.2 Univariate analysis -- categorical variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df_eda['CENTER'].value_counts().plot(kind='bar', ax=axes[0], color='#2563eb', edgecolor='white', rot=30)
axes[0].set_title('Patients by hospital center')
axes[0].set_ylabel('Count')

df_eda['CYTO_CLASS'].value_counts().plot(kind='bar', ax=axes[1], color='#7c3aed', edgecolor='white', rot=30)
axes[1].set_title('Cytogenetic risk groups (ELN 2022)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Events: {df_eda[TARGET_EVENT].sum()} deaths / {len(df_eda)} patients ({df_eda[TARGET_EVENT].mean():.1%})')
print(df_eda['CYTO_CLASS'].value_counts())

### 2.3 Cytogenetics -- ISCN classification

The raw `CYTOGENETICS` column contains 1194 unique ISCN karyotype strings. The `cytogenetic_group` function (`src/features.py`) maps them to 10 clinical risk groups following ELN 2022 standards.

| Group | Prognosis |
|---|---|
| APL t(15;17), inv(16), t(8;21) | Favourable |
| Normal, Trisomy 8, 5q deletion | Intermediate |
| Monosomy 7, Complex | Adverse |

In [ ]:
print(f'Unique CYTOGENETICS values: {df_eda["CYTOGENETICS"].nunique()}')
print(f'Missing: {df_eda["CYTOGENETICS"].isna().sum()}')
display(df_eda['CYTO_CLASS'].value_counts())

### 2.4 Continuous hematological variables

In [ ]:
n = len(CONTINUOUS_VARS)
fig, axes = plt.subplots(n, 2, figsize=(13, 4 * n))

for i, col in enumerate(CONTINUOUS_VARS):
    s = df_eda[col].dropna()
    axes[i, 0].hist(s, bins=40, color='#2563eb', edgecolor='white', alpha=0.85)
    axes[i, 0].set_title(f'{col} -- distribution')
    axes[i, 1].boxplot(s, vert=False, patch_artist=True,
                       boxprops=dict(facecolor='#93c5fd', color='#2563eb'),
                       medianprops=dict(color='#dc2626', lw=2))
    axes[i, 1].set_title(f'{col} -- boxplot')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'continuous_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

rows = []
for col in CONTINUOUS_VARS:
    s = df_eda[col].dropna()
    rows.append({'Variable': col, 'Mean': round(s.mean(),2), 'Std': round(s.std(),2),
                 'Min': s.min(), 'Median': s.median(), 'Max': s.max(),
                 'Missing': df_eda[col].isna().sum()})
display(pd.DataFrame(rows).set_index('Variable'))

### 2.5 Multivariate analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr = df_eda[CONTINUOUS_VARS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            vmin=-1, vmax=1, linewidths=0.5, ax=axes[0])
axes[0].set_title('Pearson correlation -- hematological variables')

sns.boxplot(data=df_eda, x='CENTER', y='OS_YEARS', ax=axes[1], palette='Blues')
axes[1].set_title('OS distribution by hospital center')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'multivariate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('WBC, ANC, MONOCYTES: strongly correlated (r>0.7) -- redundant information.')
print('BM_BLAST, HB, PLT: weakly correlated -- complementary signal.')

### 2.6 Statistical hypothesis tests

| Test | Question | Method |
|---|---|---|
| T1 | Does hospital center influence survival? | Multivariate log-rank |
| T2 | Do continuous variables predict survival? | Univariate Cox PH |
| T3 | Do cytogenetic groups have different survival? | Log-rank + KM |

In [ ]:
# T1: Center effect
time_col    = 'OS_YEARS'
censure_col = 'OS_STATUS'

kmf = KaplanMeierFitter()
ax  = None
for g, dfg in df_eda.dropna(subset=[time_col, censure_col, 'CENTER']).groupby('CENTER'):
    kmf.fit(dfg[time_col], dfg[censure_col], label=str(g))
    ax = kmf.plot(ax=ax, ci_show=False, show_censors=False)

plt.title('Kaplan-Meier survival curves by hospital center')
plt.xlabel('Time (years)')
plt.ylabel('Survival probability')
plt.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), frameon=False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'km_by_center.png', dpi=150, bbox_inches='tight')
plt.show()

df_valid = df_eda.dropna(subset=[time_col, censure_col, 'CENTER'])
res = multivariate_logrank_test(df_valid[time_col], df_valid['CENTER'], df_valid[censure_col])
print('Log-rank (center):', res.summary[['test_statistic', 'p']].round(4).to_string())
print('Result: chi2=95.99, p=3.21e-11 -- center significantly influences survival.')

In [ ]:
# T2: Univariate Cox
df_cox = df_eda[[time_col, censure_col] + CONTINUOUS_VARS].dropna().copy()
cph    = CoxPHFitter()
rows   = []

for v in CONTINUOUS_VARS:
    tmp = df_cox[[time_col, censure_col]].copy()
    tmp[v] = (df_cox[v] - df_cox[v].mean()) / df_cox[v].std()
    cph.fit(tmp, duration_col=time_col, event_col=censure_col)
    rows.append({'Variable': v,
                 'HR (1 SD)': round(float(np.exp(cph.params_[v])), 3),
                 'p-value':   round(float(cph.summary.loc[v, 'p']), 4)})

display(pd.DataFrame(rows).sort_values('HR (1 SD)', ascending=False))
print('HR > 1 (risk): BM_BLAST, WBC, ANC, MONOCYTES')
print('HR < 1 (protective): HB, PLT -- all significant p < 0.05')

In [ ]:
# T3: Cytogenetic groups
kmf      = KaplanMeierFitter()
df_valid = df_eda.dropna(subset=[time_col, censure_col, 'CYTO_CLASS'])

fig, ax = plt.subplots(figsize=(10, 5))
for g, dfg in df_valid.groupby('CYTO_CLASS'):
    kmf.fit(dfg[time_col], dfg[censure_col], label=g)
    kmf.plot_survival_function(ax=ax, ci_show=False, show_censors=True,
                                censor_styles={'ms': 4, 'marker': '|'})

plt.title('Kaplan-Meier survival curves by cytogenetic class')
plt.xlabel('Time (years)')
plt.ylabel('Survival probability')
plt.legend(loc='upper right', frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'km_by_cyto_class.png', dpi=150, bbox_inches='tight')
plt.show()

res = multivariate_logrank_test(df_valid[time_col], df_valid['CYTO_CLASS'], df_valid[censure_col])
print(f'Log-rank p-value: {res.p_value:.2e}')
print('Monosomy 7 / Complex: worst prognosis (HR ~2.3 / 1.95)')
print('APL t(15;17) / Normal: best prognosis')

## 3. EDA -- Molecular data <a id='3-eda-molecular'></a>

In [ ]:
display(df_molecular.head(10))
print(f'Shape: {df_molecular.shape} -- one row per mutation, multiple per patient')
print(f'ID unique in molecular: {df_molecular["ID"].is_unique}')

In [ ]:
mut_per_patient = df_molecular.groupby('ID').size()
display(mut_per_patient.describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

mut_per_patient.hist(bins=50, ax=axes[0], color='#2563eb', edgecolor='white')
axes[0].set_title('Mutations per patient')

df_molecular['GENE'].value_counts().head(20).plot(kind='bar', ax=axes[1], color='#7c3aed', edgecolor='white', rot=45)
axes[1].set_title('Top 20 mutated genes')

df_molecular['VAF'].dropna().hist(bins=50, ax=axes[2], color='#16a34a', edgecolor='white')
axes[2].set_title('VAF distribution')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'molecular_eda.png', dpi=150, bbox_inches='tight')
plt.show()

display(df_molecular['EFFECT'].value_counts().head(10))

## 4. Preprocessing <a id='4-preprocessing'></a>

**Design principles:**
- No data leakage: preprocessing estimated on training data only
- Clinically plausible outliers kept (extreme values are real AML presentations)
- KNN imputation: robust for correlated hematological variables
- Log1p transformation: applied to right-skewed variables

### 4.1 Outlier analysis

In [ ]:
print('IQR outlier summary:')
display(summarize_outliers(df_eda))
print('Decision: extreme values are clinically plausible -- all observations retained.')

### 4.2 Missing values & dataset construction

In [ ]:
df_y_clean = df_y.dropna(subset=['OS_YEARS', 'OS_STATUS']).copy()
df_model   = df_clinical.merge(df_y_clean, on='ID', how='inner').copy()
df_model['CYTO_CLASS'] = df_model['CYTOGENETICS'].apply(cytogenetic_group)

num_cols_clin = ['BM_BLAST', 'WBC', 'ANC', 'MONOCYTES', 'HB', 'PLT']
cat_cols      = ['CENTER', 'CYTO_CLASS']

print('Missing values per column:')
display(df_model.isna().sum().rename('missing'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
msno.bar(df_model, ax=axes[0], color='#2563eb')
msno.heatmap(df_model, ax=axes[1])
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Remove rows with more than 3 missing values among clinical features
row_missing = df_model[num_cols_clin + ['CYTO_CLASS']].isna().sum(axis=1)
df_imp      = df_model[row_missing <= 3].copy()
print(f'Kept: {len(df_imp)} / {len(df_model)} patients')

### 4.3 Preprocessing pipelines

In [ ]:
# Clinical pipeline
def log1p_cols(X):
    X = X.copy()
    for c in ['WBC', 'ANC', 'MONOCYTES', 'PLT']:
        if c in X.columns:
            X[c] = np.log1p(X[c])
    return X

num_pipe = Pipeline([
    ('log1p',  FunctionTransformer(log1p_cols, feature_names_out='one-to-one')),
    ('scaler', StandardScaler()),
    ('knn',    KNNImputer(n_neighbors=5, weights='distance')),
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Other')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])

preprocess_clin = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols_clin),
    ('cat', cat_pipe, cat_cols),
])

print('Clinical pipeline built.')

### 4.4 Molecular feature engineering

Patient-level aggregation from mutation-level data:
- **Mutational burden**: N_MUT, N_GENES
- **VAF/Depth statistics**: VAF_MEAN, VAF_MAX, DEPTH_MEAN, DEPTH_MAX
- **Top-K gene binary indicators**: presence/absence of the K most frequent genes
- **Mutation effect counts**: top 10 effect types

K=30 was selected via cross-validation (K=10, 20, 30 gave similar CV scores ~0.740).

In [ ]:
def make_molecular_features(df_mol, K=30, top_effects_n=10):
    df_mol = df_mol.copy()
    top_effects = df_mol['EFFECT'].value_counts().head(top_effects_n).index
    top_genes   = df_mol['GENE'].value_counts().head(K).index

    mol_agg = (
        df_mol.groupby('ID')
        .agg(N_MUT=('GENE','size'), N_GENES=('GENE','nunique'),
             VAF_MEAN=('VAF','mean'), VAF_MAX=('VAF','max'),
             DEPTH_MEAN=('DEPTH','mean'), DEPTH_MAX=('DEPTH','max'))
        .reset_index()
    )

    df_mol['EFFECT_GRP'] = df_mol['EFFECT'].where(df_mol['EFFECT'].isin(top_effects), 'OTHER_EFFECT')
    effect_counts = (
        df_mol.pivot_table(index='ID', columns='EFFECT_GRP', values='GENE',
                           aggfunc='size', fill_value=0)
        .add_prefix('EFFECT_').reset_index()
    )

    gene_flags = (
        df_mol.assign(flag=1)
        .pivot_table(index='ID', columns='GENE', values='flag', aggfunc='max', fill_value=0)
        .reindex(columns=top_genes, fill_value=0)
        .add_prefix('GENE_').reset_index()
    )

    df_features = mol_agg.merge(effect_counts, on='ID', how='left').merge(gene_flags, on='ID', how='left')
    df_features['HAS_MOL_DATA'] = 1
    df_features = df_features.fillna(0)

    return df_features, list(top_genes), list(top_effects)

# Build training molecular features with K=30
df_mol_features, top_genes, top_effects = make_molecular_features(df_molecular, K=30)

df_train_full = df_imp.merge(df_mol_features, on='ID', how='left')
mol_cols      = [c for c in df_mol_features.columns if c != 'ID']
df_train_full[mol_cols] = df_train_full[mol_cols].fillna(0)
df_train_full['HAS_MOL_DATA'] = df_train_full['HAS_MOL_DATA'].astype(int)

num_cols_full  = num_cols_clin + mol_cols
feat_cols_full = num_cols_full + cat_cols

print(f'Training dataset (full): {df_train_full.shape}')
print(f'Features: {len(feat_cols_full)} total ({len(num_cols_clin)} clinical + {len(mol_cols)} molecular + {len(cat_cols)} categorical)')

In [ ]:
def log1p_full(X):
    X = X.copy()
    for c in ['WBC','ANC','MONOCYTES','PLT','N_MUT','N_GENES','DEPTH_MAX','DEPTH_MEAN']:
        if c in X.columns:
            X[c] = np.log1p(X[c])
    return X

num_pipe_full = Pipeline([
    ('log1p',  FunctionTransformer(log1p_full, feature_names_out='one-to-one')),
    ('scaler', StandardScaler()),
    ('knn',    KNNImputer(n_neighbors=5, weights='distance')),
])

preprocess_full = ColumnTransformer(transformers=[
    ('num', num_pipe_full, num_cols_full),
    ('cat', cat_pipe,      cat_cols),
])

print('Full pipeline built.')

## 5. Modelling & evaluation <a id='5-modelling'></a>

**Strategy:**
- GridSearchCV with 5-fold cross-validation
- Models: regularised Cox PH vs Random Survival Forest
- Selection via `pick_config`: robust config closest to best CV score within 1 std, with smallest train/val gap

In [ ]:
def cindex_scorer(estimator, X, y):
    pred = estimator.predict(X)
    return concordance_index_censored(y['event'], y['time'], pred)[0]


def run_rsf_gridsearch(df_train, feat_cols, preprocess, n_jobs=2, top_n=15):
    X = df_train[feat_cols].copy()
    y = Surv.from_arrays(event=df_train['OS_STATUS'].astype(bool).values,
                         time=df_train['OS_YEARS'].values)

    pipe = Pipeline([('prep', preprocess), ('model', CoxPHSurvivalAnalysis())])
    cv   = KFold(n_splits=5, shuffle=True, random_state=42)

    param_grid = [
        {'model': [CoxPHSurvivalAnalysis()], 'model__alpha': [1e-4, 1e-3, 1e-2]},
        {
            'model':                    [RandomSurvivalForest(random_state=42, n_jobs=n_jobs)],
            'model__n_estimators':      [200, 400],
            'model__min_samples_leaf':  [5, 10, 20],
            'model__min_samples_split': [10, 20],
            'model__max_features':      ['sqrt', 0.5, 0.3],
        },
    ]

    gs = GridSearchCV(pipe, param_grid=param_grid, scoring=cindex_scorer,
                      cv=cv, n_jobs=n_jobs, refit=True, return_train_score=True, verbose=1)
    gs.fit(X, y)

    print(f'Best CV C-index : {gs.best_score_:.4f}')
    print(f'Best model      : {type(gs.best_estimator_.named_steps["model"]).__name__}')
    print(f'Best params     : {gs.best_params_}')

    keep = ['mean_test_score','std_test_score','mean_train_score','std_train_score',
            'rank_test_score','params']
    results_df = pd.DataFrame(gs.cv_results_)[keep].sort_values('mean_test_score', ascending=False)
    display(results_df.head(top_n))

    return gs, gs.best_estimator_, gs.best_score_, results_df


def pick_config(df_scores):
    df = df_scores.copy()
    df['gap'] = df['mean_train_score'] - df['mean_test_score']
    best = df['mean_test_score'].max()
    std  = df.loc[df['mean_test_score'].idxmax(), 'std_test_score']
    cand = df[df['mean_test_score'] >= best - std].copy()
    return cand.sort_values(['gap','mean_test_score'], ascending=[True,False]).iloc[0]

print('Functions defined.')

In [ ]:
# Clinical-only model
print('=' * 60)
print('CLINICAL MODEL')
print('=' * 60)

feat_cols_clin = num_cols_clin + cat_cols
gs_clin, best_clin, score_clin, df_scores_clin = run_rsf_gridsearch(
    df_imp, feat_cols_clin, preprocess_clin, n_jobs=2, top_n=15
)

In [ ]:
# Clinical + Molecular model
print('=' * 60)
print('FULL MODEL (clinical + molecular)')
print('=' * 60)

gs_full, best_full, score_full, df_scores_full = run_rsf_gridsearch(
    df_train_full, feat_cols_full, preprocess_full, n_jobs=2, top_n=15
)

print('=' * 60)
print(f'Clinical only CV C-index  : {score_clin:.4f}')
print(f'Clinical + Molecular      : {score_full:.4f}  (+{score_full - score_clin:.4f})')
print('=' * 60)

In [ ]:
# Robust model selection
row_clin = pick_config(df_scores_clin)
row_full = pick_config(df_scores_full)

print('Selected clinical config:')
display(row_clin[['mean_test_score','std_test_score','mean_train_score','gap','params']])

print('Selected full config:')
display(row_full[['mean_test_score','std_test_score','mean_train_score','gap','params']])

In [ ]:
# Save gridsearch scores
df_scores_clin.to_csv(FIGURES_DIR.parent / 'gridsearch_scores_clinical.csv', index=False)
df_scores_full.to_csv(FIGURES_DIR.parent / 'gridsearch_scores_full.csv',     index=False)
print('Gridsearch scores saved to results/')

## 6. Final fitting & submission <a id='6-final'></a>

In [ ]:
# Final clinical model
X_clin = df_imp[feat_cols_clin].copy()
y_clin = Surv.from_arrays(event=df_imp['OS_STATUS'].astype(bool).values,
                           time=df_imp['OS_YEARS'].values)

pipe_final_clin = clone(gs_clin.estimator).set_params(**row_clin['params'])
pipe_final_clin.fit(X_clin, y_clin)

c_train_clin = concordance_index_censored(
    y_clin['event'], y_clin['time'], pipe_final_clin.predict(X_clin))[0]
print(f'Clinical model -- train C-index: {c_train_clin:.4f}')

In [ ]:
# Final full model
X_full = df_train_full[feat_cols_full].copy()
y_full = Surv.from_arrays(event=df_train_full['OS_STATUS'].astype(bool).values,
                           time=df_train_full['OS_YEARS'].values)

pipe_final_full = clone(gs_full.estimator).set_params(**row_full['params'])
pipe_final_full.fit(X_full, y_full)

c_train_full = concordance_index_censored(
    y_full['event'], y_full['time'], pipe_final_full.predict(X_full))[0]
print(f'Full model -- train C-index: {c_train_full:.4f}')

In [ ]:
# Save all models
joblib.dump(pipe_final_clin,  CLINICAL_RSF)
joblib.dump(gs_clin,          CLINICAL_PIPELINE)
joblib.dump(pipe_final_full,  FULL_RSF)
joblib.dump(gs_full,          FULL_PIPELINE)
joblib.dump(feat_cols_full,   FULL_SELECTED_FEATURES)
joblib.dump({'top_genes': top_genes, 'top_effects': top_effects}, FULL_MOLECULAR_FEATURES)

print('Models saved:')
check_models()

In [ ]:
# Build test features
df_test_clin = pd.read_csv(CLINICAL_TEST)
df_test_mol  = pd.read_csv(MOLECULAR_TEST)

df_test_clin['CYTO_CLASS'] = df_test_clin['CYTOGENETICS'].apply(cytogenetic_group)

def build_mol_features(df_mol, top_genes, top_effects):
    df_mol = df_mol.copy()
    mol_agg = (
        df_mol.groupby('ID')
        .agg(N_MUT=('GENE','size'), N_GENES=('GENE','nunique'),
             VAF_MEAN=('VAF','mean'), VAF_MAX=('VAF','max'),
             DEPTH_MEAN=('DEPTH','mean'), DEPTH_MAX=('DEPTH','max'))
        .reset_index()
    )
    df_mol['EFFECT_GRP'] = df_mol['EFFECT'].where(df_mol['EFFECT'].isin(top_effects), 'OTHER_EFFECT')
    effect_counts = (
        df_mol.pivot_table(index='ID', columns='EFFECT_GRP', values='GENE',
                           aggfunc='size', fill_value=0)
        .add_prefix('EFFECT_').reset_index()
    )
    gene_flags = (
        df_mol.assign(flag=1)
        .pivot_table(index='ID', columns='GENE', values='flag', aggfunc='max', fill_value=0)
        .reindex(columns=top_genes, fill_value=0)
        .add_prefix('GENE_').reset_index()
    )
    df_feat = mol_agg.merge(effect_counts, on='ID', how='left').merge(gene_flags, on='ID', how='left')
    df_feat['HAS_MOL_DATA'] = 1
    df_feat = df_feat.fillna(0)
    return df_feat

df_mol_test  = build_mol_features(df_test_mol, top_genes, top_effects)
df_test_full = df_test_clin.merge(df_mol_test, on='ID', how='left')

for c in mol_cols:
    if c not in df_test_full.columns:
        df_test_full[c] = 0
df_test_full[mol_cols]       = df_test_full[mol_cols].fillna(0)
df_test_full['HAS_MOL_DATA'] = df_test_full['HAS_MOL_DATA'].astype(int)

print(f'Test full shape: {df_test_full.shape}')
missing = [c for c in feat_cols_full if c not in df_test_full.columns]
print(f'Missing columns in test: {missing if missing else "none"}')

In [ ]:
# Generate submission
X_test      = df_test_full[feat_cols_full].copy()
risk_scores = pipe_final_full.predict(X_test)

submission = pd.DataFrame({'ID': df_test_full['ID'].values, 'risk_score': risk_scores})
submission.to_csv('submission_full.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(f'NaN in risk scores: {np.isnan(risk_scores).sum()}')
print(f'Score range: [{risk_scores.min():.4f}, {risk_scores.max():.4f}]')
display(submission.head())

---

## Summary

| Model | CV C-index | Test platform |
|---|---|---|
| RSF -- Clinical only | 0.7138 | -- |
| RSF -- Clinical + Molecular | 0.7419 | **0.7524** |

**Key findings:**
- Cytogenetic classification (ELN 2022) is the dominant prognostic feature
- Molecular features add +2.8 points over clinical data alone
- Robust model selection via `pick_config` improved generalisation from 0.742 to 0.752
- RSF captures non-linear interactions that Cox PH cannot model